In [ ]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import numpy as np

def count_active_duids_per_day(input_dir, output_dir=None, plot_file=None, csv_file=None):
    """
    Count unique/active DUIDs per day from parquet files and create a time series plot
    
    Parameters:
    -----------
    input_dir : str
        Directory containing the parquet files with bid data
    output_dir : str, optional
        Directory to save output files (defaults to input_dir if None)
    plot_file : str, optional
        Filename for the plot (defaults to 'active_duids_per_day_plot.png')
    csv_file : str, optional
        Filename for the CSV data (defaults to 'active_duids_per_day_counts.csv')
    """
    print(f"Counting active DUIDs per day from files in {input_dir}...")
    
    # Set default output directory and filenames if not provided
    if output_dir is None:
        output_dir = input_dir
    if plot_file is None:
        plot_file = "active_duids_per_day_plot.png"
    if csv_file is None:
        csv_file = "active_duids_per_day_counts.csv"
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Get list of all parquet files
    file_list = glob.glob(os.path.join(input_dir, "*.parquet"))
    file_list.sort()
    
    if not file_list:
        print(f"No parquet files found in {input_dir}")
        return
    
    print(f"Found {len(file_list)} parquet files to process")
    
    # Initialize a dictionary to store DUID counts per day
    duids_by_day = {}
    duids_by_day_bidtype = {}  # To track DUIDs by bid type
    all_duids = set()  # Track all unique DUIDs across the entire dataset
    
    # Process each file
    for idx, file_path in enumerate(file_list, start=1):
        try:
            # Print progress every 10 files
            if idx % 10 == 0 or idx == 1 or idx == len(file_list):
                print(f"Processing file {idx}/{len(file_list)}: {os.path.basename(file_path)}")
            
            # Read the parquet file
            df = pd.read_parquet(file_path)
            
            if 'SETTLEMENTDATE' not in df.columns or 'DUID' not in df.columns:
                print(f"  Warning: Required columns not found in {os.path.basename(file_path)}")
                continue
            
            # Extract the date part from SETTLEMENTDATE
            df['DATE'] = pd.to_datetime(df['SETTLEMENTDATE']).dt.date
            
            # Track all unique DUIDs
            all_duids.update(df['DUID'].unique())
            
            # Count unique DUIDs per day
            unique_duids_per_day = df.groupby('DATE')['DUID'].nunique()
            
            # Update the overall counts
            for date, count in unique_duids_per_day.items():
                date_str = str(date)
                if date_str in duids_by_day:
                    # Update with max since we could have the same date in multiple files
                    duids_by_day[date_str] = max(duids_by_day[date_str], count)
                else:
                    duids_by_day[date_str] = count
            
            # Count DUIDs by bid type if available
            if 'BIDTYPE' in df.columns:
                # Count unique DUIDs by date and bid type
                duid_type_counts = df.groupby(['DATE', 'BIDTYPE'])['DUID'].nunique().reset_index(name='UNIQUE_DUIDS')
                
                for _, row in duid_type_counts.iterrows():
                    date_str = str(row['DATE'])
                    bid_type = row['BIDTYPE']
                    count = row['UNIQUE_DUIDS']
                    
                    if date_str not in duids_by_day_bidtype:
                        duids_by_day_bidtype[date_str] = {}
                    
                    duids_by_day_bidtype[date_str][bid_type] = count
                
                # Also collect which specific DUIDs are present for each day and type
                # This helps identify overlap between RAISEREG and LOWERREG
                grouped = df.groupby(['DATE', 'BIDTYPE'])['DUID'].apply(set).reset_index()
                
                # Store the actual DUID sets
                for _, row in grouped.iterrows():
                    date_str = str(row['DATE'])
                    bid_type = row['BIDTYPE']
                    duid_set = row['DUID']
                    
                    if date_str not in duids_by_day_bidtype:
                        duids_by_day_bidtype[date_str] = {}
                        
                    duids_by_day_bidtype[date_str][f"{bid_type}_set"] = duid_set
        
        except Exception as e:
            print(f"  Error processing {os.path.basename(file_path)}: {str(e)}")
    
    if not duids_by_day:
        print("No DUID data found in the files")
        return
    
    # Convert to DataFrame for easier handling
    dates = sorted(duids_by_day.keys())
    counts = [duids_by_day[date] for date in dates]
    
    count_df = pd.DataFrame({
        'date': dates,
        'total_active_duids': counts
    })
    
    # Convert date strings to datetime for proper sorting
    count_df['date'] = pd.to_datetime(count_df['date'])
    count_df = count_df.sort_values('date')
    
    # Add bid type columns if available
    if duids_by_day_bidtype:
        # Find all unique bid types
        all_bid_types = set()
        for date_types in duids_by_day_bidtype.values():
            all_bid_types.update([k for k in date_types.keys() if not k.endswith('_set')])
        
        # Add columns for each bid type
        for bid_type in all_bid_types:
            count_df[f'duids_{bid_type}'] = count_df['date'].astype(str).map(
                lambda date_str: duids_by_day_bidtype.get(date_str, {}).get(bid_type, 0)
            )
        
        # Calculate DUIDs that provide both services (if both RAISEREG and LOWERREG present)
        if 'RAISEREG' in all_bid_types and 'LOWERREG' in all_bid_types:
            count_df['duids_both_services'] = count_df['date'].astype(str).apply(
                lambda date_str: len(
                    duids_by_day_bidtype.get(date_str, {}).get('RAISEREG_set', set()) & 
                    duids_by_day_bidtype.get(date_str, {}).get('LOWERREG_set', set())
                ) if date_str in duids_by_day_bidtype else 0
            )
    
    # Save to CSV
    csv_path = os.path.join(output_dir, csv_file)
    count_df.to_csv(csv_path, index=False)
    print(f"Saved DUID counts to {csv_path}")
    
    # Create plot
    create_time_series_plot(count_df, output_dir, plot_file, total_unique_duids=len(all_duids))
    
    return count_df

def create_time_series_plot(count_df, output_dir, plot_file, total_unique_duids=None):
    """
    Create a time series plot of active DUIDs per day
    """
    plt.figure(figsize=(12, 6))
    
    # Set style
    sns.set_style("whitegrid")
    
    # Plot total active DUIDs
    sns.lineplot(
        data=count_df,
        x='date',
        y='total_active_duids',
        linewidth=2,
        marker='o',
        markersize=4,
        label='Total Active DUIDs'
    )
    
    # Plot bid types if available
    bid_type_columns = [col for col in count_df.columns if col.startswith('duids_') and col != 'duids_both_services']
    both_services_column = 'duids_both_services' if 'duids_both_services' in count_df.columns else None
    
    if bid_type_columns:
        # Create a separate plot for bid types
        plt.figure(figsize=(12, 6))
        
        plot_df = count_df.copy()
        
        # Plot each bid type
        for col in bid_type_columns:
            bid_type = col.replace('duids_', '')
            sns.lineplot(
                data=plot_df,
                x='date',
                y=col,
                linewidth=2,
                marker='o',
                markersize=4,
                label=f'{bid_type}'
            )
        
        # Plot DUIDs providing both services if available
        if both_services_column is not None:
            sns.lineplot(
                data=plot_df,
                x='date',
                y=both_services_column,
                linewidth=2,
                marker='o',
                markersize=4,
                linestyle='--',
                color='green',
                label='Both Services'
            )
        
        plt.title('Number of Active DUIDs per Day by Bid Type')
        plt.xlabel('Date')
        plt.ylabel('Number of DUIDs')
        plt.xticks(rotation=45)
        plt.legend()
        plt.tight_layout()
        
        # Save bid types plot
        bid_types_plot_path = os.path.join(output_dir, 'active_duids_by_type_plot.png')
        plt.savefig(bid_types_plot_path, dpi=300)
        print(f"Saved DUID types plot to {bid_types_plot_path}")
        
        # Return to the total DUIDs plot
        plt.figure(1)
    
    # Customize total DUIDs plot
    plt.title('Total Number of Active DUIDs per Day')
    plt.xlabel('Date')
    plt.ylabel('Number of Active DUIDs')
    plt.xticks(rotation=45)
    
    # Add rolling average
    if len(count_df) > 7:
        rolling_avg = count_df['total_active_duids'].rolling(window=7).mean()
        plt.plot(count_df['date'], rolling_avg, 'r--', linewidth=2, label='7-day Moving Average')
    
    # Add reference line for total unique DUIDs if provided
    if total_unique_duids:
        plt.axhline(y=total_unique_duids, linestyle=':', color='green', 
                   label=f'Total Unique DUIDs ({total_unique_duids})')
    
    plt.legend()
    plt.tight_layout()
    
    # Save the plot
    plot_path = os.path.join(output_dir, plot_file)
    plt.savefig(plot_path, dpi=300)
    print(f"Saved plot to {plot_path}")
    
    # Show plot stats
    min_date = count_df['date'].min()
    max_date = count_df['date'].max()
    total_days = (max_date - min_date).days + 1
    days_with_data = len(count_df)
    
    print(f"\nPlot statistics:")
    print(f"Date range: {min_date.date()} to {max_date.date()} ({total_days} days)")
    print(f"Days with DUID data: {days_with_data}")
    print(f"Total unique DUIDs across all days: {total_unique_duids}")
    print(f"Maximum active DUIDs in a day: {count_df['total_active_duids'].max()} on {count_df.loc[count_df['total_active_duids'].idxmax(), 'date'].date()}")
    print(f"Minimum active DUIDs in a day: {count_df['total_active_duids'].min()} on {count_df.loc[count_df['total_active_duids'].idxmin(), 'date'].date()}")
    print(f"Average active DUIDs per day: {count_df['total_active_duids'].mean():.1f}")

def analyze_duid_participation(count_df, input_dir, output_dir):
    """
    Additional analysis of DUID participation patterns
    """
    print("\nAnalyzing DUID participation patterns...")
    
    # Check if we have the necessary columns for this analysis
    if not all(col in count_df.columns for col in ['duids_RAISEREG', 'duids_LOWERREG', 'duids_both_services']):
        print("  Missing required columns for participation analysis")
        return
    
    # Calculate percentages of DUIDs providing both services
    count_df['pct_raise_also_lower'] = (count_df['duids_both_services'] / count_df['duids_RAISEREG'] * 100).round(1)
    count_df['pct_lower_also_raise'] = (count_df['duids_both_services'] / count_df['duids_LOWERREG'] * 100).round(1)
    
    # Calculate averages over time
    avg_pct_raise_also_lower = count_df['pct_raise_also_lower'].mean()
    avg_pct_lower_also_raise = count_df['pct_lower_also_raise'].mean()
    
    print(f"  Average % of RAISEREG DUIDs also providing LOWERREG: {avg_pct_raise_also_lower:.1f}%")
    print(f"  Average % of LOWERREG DUIDs also providing RAISEREG: {avg_pct_lower_also_raise:.1f}%")
    
    # Create a plot of these percentages over time
    plt.figure(figsize=(12, 6))
    
    sns.lineplot(
        data=count_df,
        x='date',
        y='pct_raise_also_lower',
        linewidth=2,
        label='% of RAISEREG DUIDs also providing LOWERREG'
    )
    
    sns.lineplot(
        data=count_df,
        x='date',
        y='pct_lower_also_raise',
        linewidth=2,
        label='% of LOWERREG DUIDs also providing RAISEREG'
    )
    
    plt.title('Percentage of DUIDs Providing Both Services')
    plt.xlabel('Date')
    plt.ylabel('Percentage (%)')
    plt.ylim(0, 100)
    plt.xticks(rotation=45)
    plt.legend()
    plt.tight_layout()
    
    # Save the plot
    plot_path = os.path.join(output_dir, 'duid_service_overlap_percentages.png')
    plt.savefig(plot_path, dpi=300)
    print(f"Saved service overlap analysis to {plot_path}")
    
    # Save the enhanced dataframe to CSV
    enhanced_csv_path = os.path.join(output_dir, 'duid_participation_analysis.csv')
    count_df.to_csv(enhanced_csv_path, index=False)

if __name__ == "__main__":
    # Directory containing the parquet files
    input_directory = "/Volumes/T7/bid-volume-filtered-3"  # Update this to your directory
    output_directory = "/Volumes/T7/bid-analysis"  # Where to save the plot and CSV
    
    # Run the analysis
    duid_counts = count_active_duids_per_day(
        input_dir=input_directory,
        output_dir=output_directory,
        plot_file="active_fcas_duids_per_day.png",
        csv_file="active_fcas_duids_per_day.csv"
    )
    
    # Run additional participation analysis if data is available
    if duid_counts is not None and 'duids_RAISEREG' in duid_counts.columns and 'duids_LOWERREG' in duid_counts.columns:
        analyze_duid_participation(duid_counts, input_directory, output_directory)